For licensing see accompanying LICENSE file.  
Copyright (C) 2025 Apple Inc. All Rights Reserved.

# Semantic Regex Consistency
Tests how similar the feature descriptions are on redundant features (i.e., features that activate on similar data / represent a similar pattern).
Simulate this by randomly sampling data from the same feature multiple times and comparing the generated feature descriptions.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent
os.chdir(ROOT) # Change working directory to project root
sys.path.insert(0, str(ROOT))

In [3]:

import methods
import features
import random
from tqdm import tqdm
from contextlib import redirect_stdout


ModuleNotFoundError: No module named 'openai'

In [ ]:
# consistency tests
num_layers = 12
num_features = 25000
selected_features = random.sample(range(num_features), 5)
feature_list = [
    features.Feature('gpt2-small', f'{l}-res-jb', i)
    for l in range(num_layers)
    for i in selected_features
]
methods_list = [
    methods.SemanticRegex(output_dir='./artifacts/experiments/temp', seed=97),
    methods.EleutherActsTop20(output_dir='./artifacts/experiments/temp', seed=97),
    methods.OAITokenActPair(output_dir='./artifacts/experiments/temp', seed=97),
    methods.MaxActivationAndLogit(output_dir='./artifacts/experiments/temp', seed=97),
]

method_descriptions = []
for method in methods_list:
    feature_descriptions = []
    for feature in tqdm(feature_list):
        descriptions = []
        try:
            for i in range(5):
                with redirect_stdout(None):
                    result = method.generate(
                        feature,
                        model_name='gpt-4o-mini',
                        n_data_examples=10,
                        n_tokens_per_sample=32,
                        show_breaks=True,
                        activation_threshold=0.3,
                        sampling_method='random',
                        logging=False
                    )
                descriptions.append(result['description']['description'])
        except Exception as e:
            if 'No activating data' in str(e):
                continue
            else:
                raise e
        feature_descriptions.append(descriptions)
    method_descriptions.append(feature_descriptions)


100%|██████████| 60/60 [06:12<00:00,  6.22s/it]


In [ ]:
feature_descriptions = []
for feature in tqdm(feature_list):
    descriptions = []
    try:
        for i in range(5):
            with redirect_stdout(None):
                result = methods_list[0].generate(
                    feature,
                    model_name='gpt-4o-mini',
                    n_data_examples=10,
                    n_tokens_per_sample=32,
                    show_breaks=True,
                    activation_threshold=0.3,
                    sampling_method='top',
                    logging=False
                )
            descriptions.append(result['description']['description'])
    except Exception as e:
        if 'No activating data' in str(e):
            continue
        else:
            raise e
    feature_descriptions.append(descriptions)
method_descriptions.append(feature_descriptions)


In [12]:
for i, m in enumerate(method_descriptions):
    unique_descriptions_per_feature = [1 - len(set(f))/len(f) for f in m]
    avg_unique = sum(unique_descriptions_per_feature) / len(unique_descriptions_per_feature)
    print(f'{methods_list[i].name}: average unique descriptions per feature: {avg_unique}')


semantic_regex: average unique descriptions per feature: 0.3355932203389831
eleuther_acts_top20: average unique descriptions per feature: 0.0
oai_token-act-pair: average unique descriptions per feature: 0.12203389830508478
np_max-act-logits: average unique descriptions per feature: 0.6305084745762713
